# 02 – Inntaksrammeverk: CSV → Bronze

Denne notebooken demonstrerer:
1. Last opp CSV-filer til `raw/`-bucketen i MinIO
2. Kjør inntaksjobb som leser CSV og skriver til `bronze/` som Delta-tabell
3. Verifiser resultatet og vis partisjonering
4. Bevis idempotens ved å re-kjøre jobben


## 1. Last opp testdata til MinIO (raw-lag)

Vi bruker `boto3` for å laste opp CSV-filene fra `/home/spark/notebooks/../data/sample/`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

In [ ]:
import boto3
from pathlib import Path

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="changeme",
)

sample_dir = Path("/home/spark/notebooks").parent / "data" / "sample"

for csv_file in sample_dir.glob("*.csv"):
    key = f"{csv_file.stem}/{csv_file.name}"
    s3.upload_file(str(csv_file), "raw", key)
    print(f"Lastet opp: s3a://raw/{key}")

## 2. Kjør inntaksjobb: employees CSV → Bronze

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("02-ingest-csv-to-bronze")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
INGESTION_DATE = "2024-01-15"
SOURCE = "s3a://raw/employees"
TARGET = "s3a://bronze/employees"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{SOURCE}/*.csv")
)

df = df.withColumns({
    "ingestion_date": F.lit(INGESTION_DATE).cast("date"),
    "_source_path":   F.lit(SOURCE),
    "_ingested_at":   F.current_timestamp(),
})

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
    .partitionBy("ingestion_date")
    .save(TARGET)
)

print(f"Skrev {df.count()} rader til {TARGET}")

## 3. Verifiser resultat

In [1]:
df_bronze = spark.read.format("delta").load(TARGET)

print(f"Rader i Bronze: {df_bronze.count()}")
df_bronze.show()
df_bronze.printSchema()

NameError: name 'spark' is not defined

In [ ]:
# Vis partisjoner
df_bronze.groupBy("ingestion_date").count().orderBy("ingestion_date").show()

## 4. Bevis idempotens — re-kjør med samme dato

`replaceWhere` erstatter partisjonen, ikke legger til. Antall rader skal ikke øke.

In [ ]:
# Re-kjør nøyaktig samme skriving
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
    .partitionBy("ingestion_date")
    .save(TARGET)
)

antall_etter = spark.read.format("delta").load(TARGET).count()
print(f"Rader etter re-kjøring: {antall_etter}  ← skal fortsatt være 8")

In [ ]:
# Delta-historikk viser begge skriveoperasjonene
from delta.tables import DeltaTable
DeltaTable.forPath(spark, TARGET).history().select("version", "timestamp", "operation").show()

In [ ]:
spark.stop()